<div align="center">

#### [S. Mussard](https://sites.google.com/view/cv-stphane-mussard/accueil "Homepage") 

</div>

<div align="center">

#### Chapter 10: Gini PCA 

</div>


<div align="center"> <a href="https://www.python.org/"><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f8/Python_logo_and_wordmark.svg/390px-Python_logo_and_wordmark.svg.png" style="max-width: 150px; display: inline" alt="Python"/></a> 

</div>


<div align="center"> </div>

<div align="center">

Cite: S. Mussard (2025), *Machine Learning with Gini Indices: Applications with Python*, Springer.  

</div>



In [ ]:
#pip install torch
#pip install torchvision

In [ ]:
from Gini_PCA import GiniPca

# Other packages
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    from OUTLIERS import smirnov_grubbs as grubbs
except ImportError:
    from outliers import smirnov_grubbs as grubbs

###  <span style="color:red">Dataset importation</span>

In [ ]:
cars = pd.read_excel("cars.xlsx",index_col=0)
cars.head()
model = GiniPca(gini_param=2, n_components=3)
x = torch.DoubleTensor(cars.values)


In [ ]:
model.ranks(x)

In [ ]:
model.gmd(x)

In [ ]:
model.scale_gini(x)

In [ ]:
z = model.scale_gini(x) # standardization of x
print ("Gini correlation :" , "\n", model.gmd(z))


In [ ]:
print ("Gini correlations :" , "\n", model.gini_correl(x))

In [ ]:
print ("Pearson correlations", "\n", torch.corrcoef(x.T))

###  <span style="color:red">Descriptive Statistics</span>


In [ ]:
cars.mean()

In [ ]:
cars.std()

In [ ]:
cars.columns

In [ ]:
boxplot = cars.boxplot(column=['Cylinder', 'Power', 'Speed', 'Weight', 'Width', 'Length'])

In [ ]:
model.optimal_gini_param(x)

In [ ]:
model = GiniPca(gini_param = 2.4, n_components = 3)
model.eigen_val(x)

In [ ]:
cars["Cylinder"].hist(bins=10)
plt.show()

###  <span style="color:red">Principal Components </span>

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale

# Graph: % of GGMD of each principal component
plt.plot(model.eigen_val(x))
plt.xlabel("Principal Components")
plt.ylabel("% of GGMD explained")
plt.show()

In [ ]:
### % Cumul GGMD

valp_ratios = torch.cumsum(model.eigen_val(x), dim = 0)
plt.bar(range(len(valp_ratios)), valp_ratios)
plt.xticks(range(len(valp_ratios)))
plt.xlabel("Principal Components")
plt.ylabel("Share of cumul GGMD %")
plt.show()


In [ ]:
### % Cumul GGMD

valp_ratios = model.eigen_val(x)
color = []
for value in valp_ratios:
    if value > 70:
        color.append('red')
    else:
        color.append('blue')
plt.bar(range(len(valp_ratios)), valp_ratios, color = color)
plt.xticks(range(len(valp_ratios)))
plt.xlabel("Principal Components")
plt.ylabel("Eigenvalues in %")
plt.show()


In [ ]:
# Projection of the points (Gini)
components = model.fit(x)[0] 
plt.figure(figsize=(10,6))
for i, j, label in zip(components[:,0], components[:,1], cars.index):
    plt.text(i, j, label, color="blue")
plt.axis((-1.5,1.5,-1,1))  
plt.axhline(0, color='red')  # horizontal axis
plt.axvline(0, color='red')  # vertical axis
plt.show()

In [ ]:
# Projection of the points (Variance)
cars_scale=scale(cars) # standardization
pca = PCA()
comp_var = pca.fit_transform(cars_scale)
# Plot
plt.figure(figsize=(10,6))
for i, j, label in zip(comp_var[:,0], comp_var[:,1], cars.index):
    plt.text(i, j, label, color="blue")
plt.axis((-6,6,-3,3)) 
plt.axhline(0, color='red')  # horizontal axis
plt.axvline(0, color='red')  # vertical axis
plt.show()

In [ ]:
# Grubbs test 10%
components_copy = components.numpy()
outliers_var = []
outliers_gini = []
for i in range (2):
    outliers_var.append(grubbs.max_test_indices(comp_var[:,i], alpha = 0.1))
    outliers_gini.append(grubbs.max_test_indices(components_copy[:,i], alpha = 0.1))
print('Atypical points in standard PCA:', outliers_var)
print('Atypical points in Gini PCA:', outliers_gini)

In [ ]:
# Grubbs test (10%)
model.outliers(x)

In [ ]:
### Box plot
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.boxplot(components_copy*(-1))
ax2.boxplot(comp_var)
plt.show()



In [ ]:
# Circle of correlations
project_var = model.gini_correl_axis(x)
# Plot
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(1, 1, 1)
for i, j, label in zip(project_var[0,:],project_var[1,:], cars.columns):
    plt.text(i, j, label)
    plt.arrow(0,0,i,j,color='gray')
plt.axis((-2.2,2.2,2.2,-2.2))
# Circle
circle = plt.Circle((0,0), radius=2.1, color='blue', fill=False)
ax.add_patch(circle)
plt.axvline(0, color='r') # vertical axis
plt.axhline(0, color='r') # horizontal axis
plt.xlabel("Axis 1")
plt.ylabel("Axis 2")
plt.show()


### ACT of variables 


In [ ]:
# Absolute contributions Axis 1: ACT of features
U = model.u_stat(x)
U_test = []
for i,value in enumerate(U[0,:]):
    if torch.abs(value) >= 1.96:
        U_test.append(cars.columns[i])
print("Significant variables on axis 1:", U_test)

In [ ]:
# Absolute contributions Axis 2: ACT of features
U = model.u_stat(x)
U_test = []
for i,value in enumerate(U[1,:]):
    if torch.abs(value) >= 1.96:
        U_test.append(cars.columns[i])
print("Significant variables on axis 1:", U_test)

### ACT of points

In [ ]:
# Axis 1: ACT of individuals
ACT = model.absolute_contrib(x)*100
list_1 = []
for i,value in enumerate(ACT[:,0]):
    if ACT[i,0] >= 4.17: 
        list_1.append(cars.index[i])
print("Significant individuals:", list_1)

In [ ]:
# Axis 2: ACT of individuals
ACT = model.absolute_contrib(x)*100
liste_2 = []
for i,value in enumerate(ACT[:,1]):
    if ACT[i,1] >= 4.17: 
        liste_2.append(cars.index[i])
print("Significant individuals:", liste_2)

In [ ]:
# Relative contributions RCT (axis 1)
RCT = model.relative_contrib(x)
print("Relative contributions:",'\n', RCT[:,0])

### Plot 3D

In [ ]:
# Plot3D

model.plot3D(cars)

###  <span style="color:red">Outliers in standard PCA </span>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from matplotlib.legend_handler import HandlerPathCollection

# Assuming X is already defined as 'components_copy'
#X = comp_var  # Keep the original data
X = components_copy 
n_outliers = 5

# Define ground truth for comparison
ground_truth = np.ones(len(X), dtype=int)
ground_truth[-n_outliers:] = -1  # Mark last `n_outliers` as outliers

# Apply Local Outlier Factor
clf = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
y_pred = clf.fit_predict(X)
n_errors = (y_pred != ground_truth).sum()
X_scores = clf.negative_outlier_factor_

# Normalize the scores for visualization
radius = (X_scores.max() - X_scores) / (X_scores.max() - X_scores.min())

# Assign colors: Inliers vs Outliers
colors = np.where(y_pred == -1, "red", "blue")  # Outliers in red, inliers in blue

# Create scatter plot with color mapping
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X[:, 0],
    X[:, 1],
    c=X_scores,  # Use LOF scores for color intensity
    cmap="coolwarm",  # Colormap for better visuals
    s=200 * radius + 10,  # Adjust marker size dynamically
    edgecolors=np.where(y_pred == -1, "black", "white"),  # Outliers get black edge
    linewidth=1.2,
    alpha=0.8,  # Add some transparency
)

# Customize axis & labels
plt.axis("tight")
plt.xlim((-2, 2))
plt.ylim((-1, 1))
plt.xlabel(f"Prediction Errors: {n_errors}")
plt.title("Local Outlier Factor (LOF) - Outlier Detection")

# Add colorbar for outlier scores
cbar = plt.colorbar(scatter)
cbar.set_label("LOF Score")

# Add legend
plt.scatter([], [], c="blue", s=100, label="Inliers")  #
plt.show()


###  <span style="color:red">MNIST dataset </span>

In [ ]:
import torchvision
import torch
import torchvision.datasets as datasets
from torchvision.transforms import ToTensor

mnist_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=None)
x_train=mnist_trainset.data.type(torch.DoubleTensor)/255

mnist_testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=None)
x_test=mnist_testset.data.type(torch.DoubleTensor)/255

x_train = torch.reshape(x_train, (x_train.shape[0], 784))
x_test = torch.reshape(x_test, (x_test.shape[0], 784))


In [ ]:
# First model
nb_comp = 7
alpha = 2
model = GiniPca(gini_param=alpha, n_components=nb_comp)
F = model.fit_inverse(x_train, x_test)
F = torch.reshape(F, (x_test.shape[0], 28, 28))

In [ ]:
# Second model
alpha1 = 3
model = GiniPca(gini_param=alpha1, n_components=nb_comp)
F1 = model.fit_inverse(x_train, x_test)
F1 = torch.reshape(F1, (x_test.shape[0], 28, 28))

In [ ]:
plt.figure(figsize=(12,8))
chiffre = 84
# Model 1
plt.subplot(1, 3, 1)
plt.imshow(F.numpy()[chiffre],
              cmap = plt.cm.gray)
plt.xlabel(f"{nb_comp} components / gini_param ={alpha}", fontsize = 14)
# Model 2
plt.subplot(1, 3, 2)
plt.imshow(F1.numpy()[chiffre],
              cmap = plt.cm.gray)
plt.xlabel(f"{nb_comp} components / gini_param = {alpha1}", fontsize = 14)
# Original
xx = torch.reshape(x_test, (10000, 28, 28))
plt.subplot(1, 3, 3)
plt.imshow(xx[chiffre], cmap=plt.cm.gray)
plt.xlabel("Original", fontsize = 14)
plt.show()


###  <span style="color:red">Fashion MNIST dataset </span>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),  
    transforms.Normalize((0.5,), (0.5,))  
    ])

# Load
train_set = torchvision.datasets.FashionMNIST(root='./data', train=True,
                                              download=True, transform=transform)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False,
                                             download=True, transform=transform)
x_train = train_set.data.unsqueeze(1).float()
x_test = test_set.data.unsqueeze(1).float()
x_train = torch.reshape(x_train, (x_train.shape[0], 784))
x_test = torch.reshape(x_test, (x_test.shape[0], 784))

In [ ]:
# Add noise in testing images
# Add Gaussian noise to the test set
noise = torch.randn(x_test.shape) * 0.3
x_test_noisy = x_test + noise
x_test_noisy = torch.clamp(x_test_noisy, 0, 1)
# Original image
plt.subplot(1, 2, 1)
xx = torch.reshape(x_test, (10000, 28, 28))
plt.imshow(xx[1], cmap=plt.cm.gray)
plt.xlabel("Original", fontsize = 14)
# Original
plt.subplot(1, 2, 2)
xx_noisy = torch.reshape(x_test_noisy, (10000, 28, 28))
plt.imshow(xx_noisy[1], cmap=plt.cm.gray)
plt.xlabel("Noisy", fontsize = 14)

In [ ]:
# First model
nb_comp = 5
alpha = 2
model = GiniPca(gini_param=alpha, n_components=nb_comp)
F = model.fit_inverse(x_train, x_test_noisy)
F = torch.reshape(F, (x_test.shape[0], 28, 28))
# Second model
alpha1 = 3
model = GiniPca(gini_param=alpha1, n_components=nb_comp)
F1 = model.fit_inverse(x_train, x_test_noisy)
F1 = torch.reshape(F1, (x_test.shape[0], 28, 28))

In [ ]:
plt.figure(figsize=(12,8))
chiffre = 1

plt.subplot(1, 3, 1)
plt.imshow(F.numpy()[chiffre],
              cmap = plt.cm.gray)
plt.xlabel(f"{nb_comp} components / gini_param ={alpha}", fontsize = 14)

# alpha = alpha1
plt.subplot(1, 3, 2)
plt.imshow(F1.numpy()[chiffre],
              cmap = plt.cm.gray)
plt.xlabel(f"{nb_comp} components / gini_param = {alpha1}", fontsize = 14)

# Originale
xx = torch.reshape(x_test_noisy, (10000, 28, 28))
plt.subplot(1, 3, 3)
plt.imshow(xx[chiffre], cmap=plt.cm.gray)
plt.xlabel("Original", fontsize = 14)


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=5)
pca.fit(x_train)
compress_images = pca.transform(x_test_noisy)
reconstruct_images = pca.inverse_transform(compress_images)
reconstruct_images.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Labels
train_labels = [label for _, label in train_set]
test_labels = [label for _, label in test_set]
# Instantation
clf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
# Fit the model on the training data
clf.fit(x_train, train_labels)


In [ ]:
# Predict
test_pred_pca = clf.predict(reconstruct_images)
accuracy_pca = accuracy_score(test_labels, test_pred_pca)

# Predict with Gini PCA
accuracy_gini = []
nu_values = []
for nu in np.arange(1, 4.5, 0.5):
    model = GiniPca(gini_param=nu, n_components=nb_comp)
    reconstruct_images = model.fit_inverse(x_train, x_test_noisy)
    y_pred = clf.predict(reconstruct_images)
    accuracy_gini.append(accuracy_score(test_labels, y_pred))
    nu_values.append(nu)



In [ ]:
# Plot
plt.figure(figsize=(6, 4))
plt.plot(nu_values, accuracy_gini, marker='o', linestyle='-', color='b', label = 'Accuracy Gini')
plt.scatter(1, accuracy_pca, color='r', label='Accuracy PCA', zorder=5) 
plt.title('Accuracy of random forest on compressed images')
plt.xlabel('nu values')
plt.ylabel('Accuracy')
plt.legend()
plt.show()